# Coffee Soil Suitability — ESP32 TinyML

This Google Colab notebook trains a **3-class** model:

- Suitable for Robusta
- Suitable for Arabica
- Not Suitable

It evaluates the model, converts it to TensorFlow Lite, and generates an installable `CoffeeSoilESP32.zip` Arduino library.

> Important: the current Arabica/Robusta labels are proxy/derived labels. For field deployment, retrain with measured altitude and temperature as additional features.


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("TensorFlow:", tf.__version__)


## 1. Upload `uganda_coffee_model_ready_3class.csv`


In [ ]:
from google.colab import files
uploaded = files.upload()
csv_name = next(n for n in uploaded if n.lower().endswith(".csv"))
df = pd.read_csv(csv_name)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Clean and select deployable features

The first ESP32 model uses only the four raw soil measurements. This keeps the embedded interface simple and avoids using engineered threshold flags as model inputs.


In [ ]:
FEATURES = ["pH","Nitrogen_percent","Phosphorus_mg_kg","Potassium_meq_100g"]
TARGET = "Coffee_Suitability_3Class"
CLASS_NAMES = ["Suitable for Robusta","Suitable for Arabica","Not Suitable"]

work = df[FEATURES + [TARGET]].copy()
for c in FEATURES:
    work[c] = pd.to_numeric(work[c], errors="coerce")
work[TARGET] = work[TARGET].astype(str).str.strip()
work = work.dropna().drop_duplicates().reset_index(drop=True)

unexpected = set(work[TARGET]) - set(CLASS_NAMES)
if unexpected:
    raise ValueError(f"Unexpected classes: {unexpected}")

print(work[TARGET].value_counts())
display(work.describe())


## 3. Exploratory analysis


In [ ]:
work[TARGET].value_counts().reindex(CLASS_NAMES).plot(kind="bar", title="Three-class distribution")
plt.ylabel("Samples")
plt.xticks(rotation=20, ha="right")
plt.show()

work[FEATURES].hist(figsize=(10,7), bins=30)
plt.tight_layout()
plt.show()

display(work.groupby(TARGET)[FEATURES].agg(["mean","std","min","max"]).round(3))


## 4. Stratified train/validation/test split and standardization


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

class_to_id = {name:i for i,name in enumerate(CLASS_NAMES)}
X = work[FEATURES].to_numpy(dtype=np.float32)
y = work[TARGET].map(class_to_id).to_numpy(dtype=np.int32)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

weights = compute_class_weight("balanced", classes=np.arange(3), y=y_train)
class_weight = {i:float(v) for i,v in enumerate(weights)}

print("Train / Validation / Test:", len(X_train), len(X_val), len(X_test))
print("Scaler mean:", scaler.mean_)
print("Scaler scale:", scaler.scale_)


## 5. Train a compact neural network for ESP32


In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-5)
]

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=250,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

model.summary()


In [ ]:
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## 6. Evaluate the held-out test set


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
probs = model.predict(X_test_s, verbose=0)
pred = np.argmax(probs, axis=1)

print(f"Test accuracy: {test_acc:.4f}")
print(classification_report(
    y_test, pred, labels=np.arange(3),
    target_names=CLASS_NAMES, digits=4, zero_division=0
))

cm = confusion_matrix(y_test, pred, labels=np.arange(3))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(xticks_rotation=25, cmap=None)
plt.show()


## 7. Convert to TensorFlow Lite and verify it


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("coffee_soil_model.tflite","wb") as f:
    f.write(tflite_model)

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

tflite_pred = []
for row in X_test_s:
    interpreter.set_tensor(inp["index"], row.reshape(1,-1).astype(inp["dtype"]))
    interpreter.invoke()
    tflite_pred.append(np.argmax(interpreter.get_tensor(out["index"])[0]))

tflite_pred = np.array(tflite_pred)
print("TFLite model size:", len(tflite_model), "bytes")
print("TFLite test accuracy:", (tflite_pred == y_test).mean())
print("Keras/TFLite agreement:", (tflite_pred == pred).mean())


## 8. Generate the installable Arduino library

The generated library uses **ArduTFLite**. Install ArduTFLite in Arduino IDE before compiling the example.


In [ ]:
from pathlib import Path
import shutil, json

L=Path("CoffeeSoilESP32")
if L.exists():
    shutil.rmtree(L)
(L/"src").mkdir(parents=True)
(L/"examples"/"CoffeeSoilPrediction").mkdir(parents=True)

hexrows=[]
for i in range(0,len(tflite_model),12):
    hexrows.append(", ".join("0x%02x" % b for b in tflite_model[i:i+12]))
model_h = '#pragma once\n#include <Arduino.h>\n\nalignas(16) const unsigned char coffee_soil_model[] = {\n  ' + ',\n  '.join(hexrows) + '\n};\nconst unsigned int coffee_soil_model_len = ' + str(len(tflite_model)) + ';\n'
(L/"src"/"coffee_soil_model.h").write_text(model_h)

header = '#pragma once\n#include <Arduino.h>\nenum CoffeeSuitabilityClass { ROBUSTA_SUITABLE=0, ARABICA_SUITABLE=1, NOT_SUITABLE=2, PREDICTION_ERROR=-1 };\nstruct CoffeePrediction { CoffeeSuitabilityClass classId; const char* label; float confidence; float probabilities[3]; };\nclass CoffeeSoilESP32 {\npublic:\n  bool begin();\n  CoffeePrediction predict(float pH, float nitrogenPercent, float phosphorusMgKg, float potassiumMeq100g);\nprivate:\n  bool ready_ = false;\n};\n'
cpp_template = '#include "CoffeeSoilESP32.h"\n#include "coffee_soil_model.h"\n#include <ArduTFLite.h>\nstatic constexpr int NIN=4, NCLASS=3;\nstatic constexpr int TENSOR_ARENA_SIZE=16*1024;\nstatic byte tensorArena[TENSOR_ARENA_SIZE];\nstatic const float MEAN[NIN]={MEANS};\nstatic const float SCALE[NIN]={SCALES};\nstatic const char* NAMES[NCLASS]={"Suitable for Robusta","Suitable for Arabica","Not Suitable"};\n\nbool CoffeeSoilESP32::begin() {\n  ready_ = modelInit(coffee_soil_model, tensorArena, TENSOR_ARENA_SIZE);\n  return ready_;\n}\n\nCoffeePrediction CoffeeSoilESP32::predict(float ph,float n,float p,float k) {\n  CoffeePrediction r;\n  r.classId=PREDICTION_ERROR; r.label="Prediction error"; r.confidence=0.0f;\n  for(int i=0;i<NCLASS;i++) r.probabilities[i]=0.0f;\n  if(!ready_) return r;\n  float raw[NIN]={ph,n,p,k};\n  for(int i=0;i<NIN;i++)\n    if(!modelSetInput((raw[i]-MEAN[i])/SCALE[i],i)) return r;\n  if(!modelRunInference()) return r;\n  int best=0; float bestP=-1.0f;\n  for(int i=0;i<NCLASS;i++) {\n    float q=modelGetOutput(i); r.probabilities[i]=q;\n    if(q>bestP){bestP=q;best=i;}\n  }\n  r.classId=(CoffeeSuitabilityClass)best;\n  r.label=NAMES[best];\n  r.confidence=bestP;\n  return r;\n}\n'
props = 'name=CoffeeSoilESP32\nversion=1.0.0\nauthor=Coffee Soil Suitability Project\nmaintainer=Coffee Soil Suitability Project\nsentence=Three-class coffee soil suitability inference for ESP32.\nparagraph=TinyML classifier for Robusta, Arabica, or unsuitable soil.\ncategory=Data Processing\narchitectures=esp32\ndepends=ArduTFLite\n'
example = '#include <CoffeeSoilESP32.h>\nCoffeeSoilESP32 coffee;\n\nvoid setup() {\n  Serial.begin(115200);\n  delay(1000);\n\n  if(!coffee.begin()) {\n    Serial.println("Model initialization failed.");\n    while(true) delay(1000);\n  }\n\n  // Replace with real sensor values in the SAME units used during training.\n  CoffeePrediction r = coffee.predict(5.8, 0.18, 35.0, 0.80);\n\n  Serial.print("Prediction: "); Serial.println(r.label);\n  Serial.print("Confidence: "); Serial.println(r.confidence, 4);\n  Serial.print("Robusta: "); Serial.println(r.probabilities[0], 4);\n  Serial.print("Arabica: "); Serial.println(r.probabilities[1], 4);\n  Serial.print("Not suitable: "); Serial.println(r.probabilities[2], 4);\n}\nvoid loop() {}\n'
readme = '# CoffeeSoilESP32\n\nInputs, in order:\n1. pH\n2. Nitrogen (%)\n3. Phosphorus (mg/kg)\n4. Potassium (meq/100g)\n\nInstall ArduTFLite (and its Chirale_TensorFlowLite dependency) in Arduino IDE.\n\nIMPORTANT: many RS485 NPK sensors report N, P and K in mg/kg. This model currently expects N in %, P in mg/kg and K in meq/100g. Verify and convert your sensor units before inference.\n\nThe current Arabica/Robusta labels are proxy labels. Add measured altitude and temperature before treating this as a field-ready agronomic classifier.\n'

means = ", ".join("%.9ff" % float(v) for v in scaler.mean_)
scales = ", ".join("%.9ff" % float(v) for v in scaler.scale_)
cpp = cpp_template.replace("{MEANS}", means).replace("{SCALES}", scales)
cpp = cpp.replace("static const float MEAN[NIN]=", "static const float MEAN[NIN]={", 1).replace(";\nstatic const float SCALE[NIN]=", "};\nstatic const float SCALE[NIN]={", 1).replace(";\nstatic const char* NAMES", "};\nstatic const char* NAMES", 1)

(L/"src"/"CoffeeSoilESP32.h").write_text(header)
(L/"src"/"CoffeeSoilESP32.cpp").write_text(cpp)
(L/"library.properties").write_text(props)
(L/"examples"/"CoffeeSoilPrediction"/"CoffeeSoilPrediction.ino").write_text(example)
(L/"README.md").write_text(readme)

metadata={
    "features": FEATURES,
    "classes": CLASS_NAMES,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "test_accuracy": float(test_acc),
    "tflite_model_bytes": len(tflite_model)
}
(L/"training_metadata.json").write_text(json.dumps(metadata, indent=2))

shutil.make_archive("CoffeeSoilESP32","zip",root_dir=".",base_dir="CoffeeSoilESP32")
print("Generated CoffeeSoilESP32.zip")


## 9. Download the generated Arduino library


In [ ]:
from google.colab import files
files.download("CoffeeSoilESP32.zip")


## Arduino installation

1. Run all notebook cells.
2. Upload `uganda_coffee_model_ready_3class.csv` when requested.
3. The final cell downloads `CoffeeSoilESP32.zip`.
4. In Arduino IDE Library Manager, install **ArduTFLite**.
5. Go to **Sketch → Include Library → Add .ZIP Library** and select `CoffeeSoilESP32.zip`.
6. Open **File → Examples → CoffeeSoilESP32 → CoffeeSoilPrediction**.

### Sensor-unit warning

Do not pass raw NPK sensor readings into the model until the sensor units are confirmed. The current training data uses N in **%**, P in **mg/kg**, and K in **meq/100g**.
